# Second-Codec Experiment: AMR-WB and Opus
## Extending the codec-degradation study beyond AMR-NB

This notebook generalises the main finding to two further codecs, addressing the most likely reviewer request. It reuses the exact protocol of the main experiment (same classifier, folds, seed, mel parameters) so results are directly comparable.

**Codecs evaluated**
- **AMR-WB** (Adaptive Multi-Rate Wideband, G.722.2) at 6.60 and 23.05 kbit/s -- the wideband sibling of AMR-NB; same ACELP family, 16 kHz sampling, ~50--7000 Hz band.
- **Opus** at 6 and 24 kbit/s -- the dominant modern IP/VoIP codec; hybrid SILK/CELT.

**Design (identical to the main experiment):** clean-trained ResNet-50 tested on codec-degraded audio (clean->codec), plus matched and codec-aware augmented training at the lower bitrate of each codec. Ten-fold cross-validation, macro-F1, fixed seed 42.

**Runtime:** designed to run overnight. Every fold is checkpointed; re-running resumes.

**Setup:** GPU T4, Internet On, and add `gurjant-us8k-official` via + Add Input.

In [ ]:
# Cell 1 - Configuration
import os, sys, json, random, subprocess, csv, warnings
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore", message="n_fft=.* is too large")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

IN      = Path("/kaggle/input")
WORK    = Path("/kaggle/working")
RESULTS = WORK / "results"; RESULTS.mkdir(parents=True, exist_ok=True)
FIGS    = RESULTS / "figs"; FIGS.mkdir(exist_ok=True)
CKPT    = RESULTS / "ckpt"; CKPT.mkdir(exist_ok=True)
PROC    = WORK / "processed"; PROC.mkdir(exist_ok=True)
MEL_CACHE = WORK / "mel_cache"; MEL_CACHE.mkdir(exist_ok=True)
TMP     = WORK / "tmp"; TMP.mkdir(exist_ok=True)

TARGET_SR = 22050; MAX_DUR = 4.0
N_MELS = 128; N_FFT = 2048; HOP = 512; N_FOLDS = 10
BATCH = 32; EPOCHS = 30; PATIENCE = 5; LR = 1e-4

# Reference clean baseline from the main experiment (identical clean->clean protocol)
A_FOLD_F1 = [0.7489, 0.7486, 0.7437, 0.7808, 0.8171,
             0.7771, 0.7788, 0.7967, 0.8190, 0.8459]
REF_A_CLEAN_MEAN = 0.786

# Codec plan. Opus is primary (works everywhere; different family from AMR-NB).
# AMR-WB is included only if its encoder is available (checked in Cell 2).
ALL_CODECS = {
    "opus":  {"sr": 16000, "codec": "libopus", "fmt": "opus",
              "low": "6k", "high": "24k",
              "low_tag": "opus_6k", "high_tag": "opus_24k"},
    "amrwb": {"sr": 16000, "codec": "libvo_amrwbenc", "fmt": "amr",
              "low": "6.60k", "high": "23.05k",
              "low_tag": "amrwb_6k6", "high_tag": "amrwb_23k"},
}
# CODECS is finalised in Cell 2b after encoder detection.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | CUDA devices: {torch.cuda.device_count()}")
assert DEVICE.type == "cuda", "Enable GPU T4 in Settings."
print("Configuration loaded.")

In [ ]:
# Cell 2 - Install codecs and detect encoder availability
subprocess.run("apt-get update -qq && apt-get install -y -qq "
               "libavcodec-extra libvo-amrwbenc0 libopus0 ffmpeg",
               shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
enc = subprocess.run(["ffmpeg","-encoders"], capture_output=True, text=True).stdout

# Opus is required and near-universal; AMR-WB is attempted but optional
OPUS_OK   = "libopus" in enc
AMRWB_OK  = "libvo_amrwbenc" in enc
print("Opus encoder present:   ", OPUS_OK)
print("AMR-WB encoder present: ", AMRWB_OK)

if not OPUS_OK:
    # try static ffmpeg as a fallback for Opus
    subprocess.run(
        "wget -q https://johnvansickle.com/ffmpeg/releases/"
        "ffmpeg-release-amd64-static.tar.xz && tar xf ffmpeg-release-amd64-static.tar.xz",
        shell=True)
    static = [d for d in os.listdir(".") if d.startswith("ffmpeg-") and d.endswith("-static")]
    if static:
        os.environ["PATH"] = os.path.abspath(static[0]) + ":" + os.environ["PATH"]
        enc = subprocess.run(["ffmpeg","-encoders"], capture_output=True, text=True).stdout
        OPUS_OK  = "libopus" in enc
        AMRWB_OK = "libvo_amrwbenc" in enc
        print("After static fallback -> Opus:", OPUS_OK, " AMR-WB:", AMRWB_OK)

assert OPUS_OK, "Opus encoder unavailable even after fallback; cannot proceed."
if not AMRWB_OK:
    print("\nNote: AMR-WB encoder not available in this environment. "
          "The notebook will run Opus only (the primary second codec) and skip AMR-WB. "
          "Opus is a different codec family from AMR-NB and is the stronger generalisation test.")

for pkg in ["librosa","soundfile","scikit-learn","scipy","tqdm"]:
    subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
import librosa, soundfile as sf
print("Dependencies installed.")

In [ ]:
# Cell 2b - Finalise the codec set based on detected encoders
CODECS = {"opus": ALL_CODECS["opus"]}
if AMRWB_OK:
    CODECS["amrwb"] = ALL_CODECS["amrwb"]
print("Codecs that will be evaluated:", list(CODECS.keys()))

In [ ]:
# Cell 3 - Discover UrbanSound8K
def find_file(pattern):
    hits = list(IN.rglob(pattern)); return hits[0] if hits else None
us8k_csv = find_file("UrbanSound8K.csv")
assert us8k_csv, "UrbanSound8K.csv not found. Add gurjant-us8k-official via + Add Input."
us8k_base = us8k_csv.parent
if not (us8k_base / "audio").exists():
    for cand in [us8k_csv.parent.parent, us8k_csv.parent.parent.parent]:
        if (cand / "audio").exists(): us8k_base = cand; break
official = (us8k_base / "audio" / "fold1").exists()
US8K_ROWS = []
with open(us8k_csv) as fh:
    for r in csv.DictReader(fh):
        fold = r["fold"]
        sub = f"audio/fold{fold}" if official else f"fold{fold}"
        US8K_ROWS.append((str(us8k_base / sub / r["slice_file_name"]), r["class"], int(r["fold"])))
CLASSES = sorted(set(c for _, c, _ in US8K_ROWS))
for fold in range(1, N_FOLDS + 1):
    s = next((p for p,_,f in US8K_ROWS if f == fold), None)
    assert s and Path(s).exists(), f"fold{fold} audio not found"
print(f"UrbanSound8K: {len(US8K_ROWS)} clips, {len(CLASSES)} classes")
assert len(US8K_ROWS) == 8732

In [ ]:
# Cell 4 - Codec degradation functions
import librosa, soundfile as sf

def encode_decode(inp, out, codec_key, bitrate):
    """Generic encode/decode round trip for AMR-WB or Opus."""
    spec = CODECS[codec_key]
    stem = str(out) + "." + spec["fmt"]
    enc = subprocess.run(
        ["ffmpeg","-y","-i",str(inp),"-ac","1","-ar",str(spec["sr"]),
         "-b:a",bitrate,"-c:a",spec["codec"],stem],
        capture_output=True, timeout=30)
    if enc.returncode != 0:
        raise RuntimeError(f"{codec_key} encode failed: {Path(inp).name}: "
                           f"{enc.stderr.decode()[-200:]}")
    dec = subprocess.run(
        ["ffmpeg","-y","-i",stem,"-ar",str(TARGET_SR),"-ac","1",str(out)],
        capture_output=True, timeout=30)
    if Path(stem).exists(): os.remove(stem)
    if dec.returncode != 0:
        raise RuntimeError(f"{codec_key} decode failed: {Path(inp).name}")

print("Codec functions defined.")

In [ ]:
# Cell 5 - Generate degraded corpora for both codecs (all four conditions)
from tqdm.auto import tqdm
from multiprocessing.pool import ThreadPool
import multiprocessing as mp

# conditions: (tag, codec_key, bitrate)
CONDITIONS = []
for ck, spec in CODECS.items():
    CONDITIONS.append((spec["low_tag"],  ck, spec["low"]))
    CONDITIONS.append((spec["high_tag"], ck, spec["high"]))
print("Conditions to generate:", [c[0] for c in CONDITIONS])

def _make_one(task):
    src, dst, ck, br = task
    if Path(dst).exists(): return "ok"
    try:
        encode_decode(src, dst, ck, br); return "ok"
    except Exception as e:
        return ("error", src, str(e))

workers = max(1, mp.cpu_count() - 1)
for tag, ck, br in CONDITIONS:
    outdir = PROC / tag
    tasks = []
    for src, _, fold in US8K_ROWS:
        fdir = outdir / f"fold{fold}"; fdir.mkdir(parents=True, exist_ok=True)
        tasks.append((src, str(fdir / Path(src).name), ck, br))
    with ThreadPool(workers) as pool:
        outcomes = list(tqdm(pool.imap(_make_one, tasks, chunksize=32),
                             total=len(tasks), desc=tag))
    errs = [o for o in outcomes if isinstance(o, tuple)]
    n_ok = len(list(outdir.rglob("*.wav")))
    if errs:
        print(f"  {tag}: {len(errs)} errors. First: {errs[0][2][:150]}")
    print(f"  {tag}: {n_ok}/{len(US8K_ROWS)} clips")
print("Corpus generation complete.")

In [ ]:
# Cell 6 - Acoustic verification (confirms each codec genuinely degrades)
import librosa, numpy as np
import random as _rnd; _rnd.seed(SEED)
def hf_fraction(y, sr=TARGET_SR, cut=3400):
    S = np.abs(librosa.stft(y)) ** 2
    f = librosa.fft_frequencies(sr=sr)
    return float(S[f >= cut].sum() / (S.sum() + 1e-12))
sample = _rnd.sample(US8K_ROWS, 30)
for tag, ck, br in CONDITIONS:
    fr = []
    for src, _, fold in sample:
        p = PROC / tag / f"fold{fold}" / Path(src).name
        if not p.exists(): continue
        y,_ = librosa.load(str(p), sr=TARGET_SR, duration=MAX_DUR)
        fr.append(hf_fraction(y))
    print(f"  {tag}: mean >3.4kHz energy fraction = {np.mean(fr):.4f}")
print("(Wideband codecs retain more high-band energy than AMR-NB; that is expected.)")

In [ ]:
# Cell 7 - Model, dataset, mel cache, training (identical protocol to main experiment)
import librosa
def to_mel(path):
    y,_ = librosa.load(path, sr=TARGET_SR, mono=True, duration=MAX_DUR)
    L = int(TARGET_SR*MAX_DUR); y = np.pad(y,(0,max(0,L-len(y))))[:L]
    m = librosa.power_to_db(librosa.feature.melspectrogram(
        y=y, sr=TARGET_SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP), ref=np.max)
    return ((m-m.mean())/(m.std()+1e-6)).astype(np.float32)

def mel_cache_path(condition, fold, name):
    d = MEL_CACHE / condition / f"fold{fold}"; d.mkdir(parents=True, exist_ok=True)
    return d / (name + ".npy")

def precompute_mels(condition, condition_dir):
    todo = []
    for src, cls, fold in US8K_ROWS:
        cp = mel_cache_path(condition, fold, Path(src).name)
        if cp.exists(): continue
        ap = src if condition_dir is None else str(Path(condition_dir)/f"fold{fold}"/Path(src).name)
        todo.append((ap, str(cp)))
    if not todo: print(f"  {condition}: cached"); return
    def _one(t):
        ap, cp = t
        try: np.save(cp, to_mel(ap)); return True
        except Exception: return False
    with ThreadPool(max(1, mp.cpu_count()-1)) as pool:
        list(tqdm(pool.imap(_one, todo, chunksize=32), total=len(todo), desc=f"mel {condition}"))
    print(f"  {condition}: cached {len(todo)}")

print("Precomputing mel-spectrograms (clean + all codec conditions)...")
precompute_mels("clean", None)
for tag, ck, br in CONDITIONS:
    precompute_mels(tag, str(PROC / tag))
print("Mel cache ready.")

class CachedMel(Dataset):
    def __init__(self, files, labels): self.files=files; self.labels=labels
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        return torch.tensor(np.load(self.files[i])).unsqueeze(0), self.labels[i]

def build_split(condition, folds):
    ci = {c:i for i,c in enumerate(CLASSES)}; files=[]; labels=[]
    for src, cls, fold in US8K_ROWS:
        if fold not in folds: continue
        cp = mel_cache_path(condition, fold, Path(src).name)
        if not cp.exists(): continue
        files.append(str(cp)); labels.append(ci[cls])
    return CachedMel(files, labels)

def build_aug_split(cond, folds):
    # union of clean and codec mels for augmented training
    ci = {c:i for i,c in enumerate(CLASSES)}; files=[]; labels=[]
    for src, cls, fold in US8K_ROWS:
        if fold not in folds: continue
        for c in ["clean", cond]:
            cp = mel_cache_path(c, fold, Path(src).name)
            if cp.exists(): files.append(str(cp)); labels.append(ci[cls])
    return CachedMel(files, labels)

class ResNet50Classifier(nn.Module):
    def __init__(self, nc):
        super().__init__()
        self.to_rgb = nn.Conv2d(1,3,1,bias=False)
        b = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
        for p in b.parameters(): p.requires_grad=False
        for p in b.layer4.parameters(): p.requires_grad=True
        self.features = nn.Sequential(*list(b.children())[:-1])
        self.embedding = nn.Sequential(nn.Flatten(), nn.Linear(2048,128), nn.ReLU(), nn.Dropout(0.3))
        self.classifier = nn.Linear(128, nc)
    def forward(self, x): return self.classifier(self.embedding(self.features(self.to_rgb(x))))

def train_eval(train_ds, test_ds, nc, seed, tag):
    ck = CKPT / f"{tag}.json"
    if ck.exists():
        r = json.load(open(ck)); print(f"  {tag}: cached F1={r['macro_f1']:.3f}"); return r
    torch.manual_seed(seed)
    model = ResNet50Classifier(nc).to(DEVICE)
    opt = torch.optim.Adam(filter(lambda p:p.requires_grad, model.parameters()), lr=LR)
    crit = nn.CrossEntropyLoss()
    nv = max(1, len(train_ds)//10); nt = len(train_ds)-nv
    tr, va = torch.utils.data.random_split(train_ds, [nt, nv],
             generator=torch.Generator().manual_seed(seed))
    tl = DataLoader(tr, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
    vl = DataLoader(va, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
    best=float("inf"); pat=0; bst=None
    for _ in range(EPOCHS):
        model.train()
        for X,y in tl:
            X,y=X.to(DEVICE),y.to(DEVICE); opt.zero_grad(); crit(model(X),y).backward(); opt.step()
        model.eval(); vloss=0.0
        with torch.no_grad():
            for X,y in vl: vloss += crit(model(X.to(DEVICE)), y.to(DEVICE)).item()
        vloss/=len(vl)
        if vloss<best: best=vloss; pat=0; bst={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat+=1
            if pat>=PATIENCE: break
    model.load_state_dict(bst); model.eval()
    pred=[]; true=[]
    for X,y in DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True):
        pred += model(X.to(DEVICE)).argmax(1).cpu().tolist(); true += y.tolist()
    r = {"macro_f1": float(f1_score(true,pred,average="macro",zero_division=0))}
    json.dump(r, open(ck,"w"))
    del model,bst; torch.cuda.empty_cache()
    print(f"  {tag}: F1={r['macro_f1']:.3f}")
    return r

print("Model and training utilities ready.")

In [ ]:
# Cell 8 - Run all codec conditions (clean-trained test; + matched/aug at low bitrate)
folds = list(range(1, N_FOLDS+1)); nc = len(CLASSES)
results = {"A_clean": {"mean": float(np.mean(A_FOLD_F1)), "std": float(np.std(A_FOLD_F1)),
                       "f1all": list(A_FOLD_F1)}}
print(f"A_clean (from main experiment): {results['A_clean']['mean']:.3f}\n")

# 1) clean-trained, tested on every codec condition
for tag, ck_, br in CONDITIONS:
    print(f"clean-trained -> tested on {tag}")
    scores=[]
    for te in folds:
        tr=[f for f in folds if f!=te]
        train_ds=build_split("clean", tr)
        test_ds =build_split(tag, [te])
        r=train_eval(train_ds, test_ds, nc, SEED, f"clean_to_{tag}_f{te}")
        scores.append(r["macro_f1"])
    results[f"clean_to_{tag}"]={"mean":float(np.mean(scores)),"std":float(np.std(scores)),"f1all":scores}
    print()

# 2) matched + augmented at the LOW bitrate of each codec (the recovery story)
for ck_key, spec in CODECS.items():
    low=spec["low_tag"]
    for regime in ["matched","aug"]:
        print(f"{regime} training on {low}")
        scores=[]
        for te in folds:
            tr=[f for f in folds if f!=te]
            if regime=="matched":
                train_ds=build_split(low, tr)
            else:
                train_ds=build_aug_split(low, tr)
            test_ds=build_split(low, [te])
            r=train_eval(train_ds, test_ds, nc, SEED, f"{regime}_{low}_f{te}")
            scores.append(r["macro_f1"])
        results[f"{regime}_{low}"]={"mean":float(np.mean(scores)),"std":float(np.std(scores)),"f1all":scores}
        print()

json.dump(results, open(RESULTS/"second_codec_results.json","w"), indent=2)
print("=== SUMMARY ===")
for k,v in results.items():
    print(f"  {k:22s} {v['mean']:.3f} +/- {v['std']:.3f}")

In [ ]:
# Cell 9 - Analysis: degradation and recovery per codec
import numpy as np, json
res = json.load(open(RESULTS/"second_codec_results.json"))
A = res["A_clean"]["mean"]
print(f"Clean baseline: {A:.3f}\n")
print(f"{'Condition':24s} {'F1':>7s} {'drop(pp)':>9s} {'recovery':>9s}")
print("-"*54)
for ck_key, spec in CODECS.items():
    for tag in [spec["low_tag"], spec["high_tag"]]:
        key=f"clean_to_{tag}"
        if key in res:
            f1=res[key]["mean"]; drop=(A-f1)*100
            print(f"{key:24s} {f1:7.3f} {drop:9.1f} {'-':>9s}")
    low=spec["low_tag"]
    base_codec=res[f"clean_to_{low}"]["mean"]
    for regime in ["matched","aug"]:
        rk=f"{regime}_{low}"
        if rk in res:
            f1=res[rk]["mean"]
            rec=100*(f1-base_codec)/(A-base_codec+1e-9)
            print(f"{rk:24s} {f1:7.3f} {(A-f1)*100:9.1f} {rec:8.0f}%")
    print()
print("Interpretation: compare each codec's clean_to_low drop against AMR-NB (41.6 pp).")
print("Wideband codecs (AMR-WB, Opus) should degrade LESS than AMR-NB,")
print("supporting the bandwidth-plus-source-model account.")

In [ ]:
# Cell 10 - Bundle
import shutil
shutil.make_archive(str(WORK/"second_codec_results"),"zip",str(RESULTS))
print("Download second_codec_results.zip from the Output tab.")
print("Files:", sorted(p.name for p in RESULTS.glob("*")))